In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ciciot2023/CICIOT23/validation/validation.csv
/kaggle/input/ciciot2023/CICIOT23/test/test.csv
/kaggle/input/ciciot2023/CICIOT23/train/train.csv


In [2]:
!pip install pennylane


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 84.0 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 33.7 MB/s eta 0:00:0000:0100:01m


In [3]:
# ==========================================
# 1. SETUP & IMPORTS
# ==========================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from torch.utils.data import Dataset, DataLoader
import joblib
import time

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. DATA LOADING & CLEANING
# ==========================================
# Load a subset because QML simulation is slow
# In real life, use more data, but for a project demo, 10k-20k is sufficient
train_path = '/kaggle/input/ciciot2023/CICIOT23/train/train.csv'
print("Loading dataset...")
df = pd.read_csv(train_path)

# Sample the data to make it manageable for Quantum Simulation
# (Quantum simulators on classical CPU/GPU struggle with millions of rows)
df = df.sample(n=15000, random_state=42) 

print(f"Dataset sampled to: {df.shape}")

# Function to clean labels (Your logic)
def clean_labels(df):
    # Map complex attacks to 1, Benign to 0 for Binary Classification
    # This makes the QML model much more stable and accurate
    df['target'] = df['label'].apply(lambda x: 0 if 'Benign' in str(x) else 1)
    return df

df = clean_labels(df)

# Separate Features and Target
X = df.drop(columns=['label', 'target'])
# Remove non-numeric columns if any exist
X = X.select_dtypes(include=[np.number])
y = df['target'].values

# ==========================================
# 3. PREPROCESSING (PCA for QUANTUM)
# ==========================================
# Quantum computers (and simulators) have few qubits. 
# We cannot feed 47 features. We must reduce to 4 (for 4 qubits).

Using device: cuda
Loading dataset...
Dataset sampled to: (15000, 47)


In [4]:


print("Preprocessing: Scaling and PCA...")

# Step A: Standard Normalization
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Step B: PCA (Reduce to 4 features)
n_qubits = 4
pca = PCA(n_components=n_qubits)
X_pca = pca.fit_transform(X_scaled)

# Step C: Rescale PCA output to [0, Pi] for Quantum Angle Embedding
# Quantum gates (RX, RY, RZ) take angles in radians.
angle_scaler = MinMaxScaler(feature_range=(0, np.pi))
X_quantum = angle_scaler.fit_transform(X_pca)

# Step D: Save these processors for the local blockchain script!
joblib.dump(scaler, 'scaler_base.pkl')
joblib.dump(pca, 'pca_model.pkl')
joblib.dump(angle_scaler, 'scaler_angle.pkl')
print("✅ Preprocessors saved.")

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X_quantum, y, test_size=0.2, random_state=42)

# PyTorch Data Loaders
class IoTDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.FloatTensor(labels).unsqueeze(1) # Binary needs (N,1)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]

train_loader = DataLoader(IoTDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(IoTDataset(X_test, y_test), batch_size=32)

# ==========================================
# 4. HYBRID QUANTUM MODEL (PyTorch)
# ==========================================
print("Building Hybrid Quantum Model...")

# A. Define Quantum Circuit (PennyLane)
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_circuit(inputs, weights):
    # Encode Data (Angle Embedding)
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    
    # Variational Layer (Learnable Weights)
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    
    # Measurement (Expectation value of PauliZ on all qubits)
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

# B. Define PyTorch Hybrid Class
class HybridModel(nn.Module):
    def __init__(self):
        super(HybridModel, self).__init__()
        
        # 1. Quantum Layer
        weight_shapes = {"weights": (3, n_qubits)} # 3 layers of entanglement
        self.q_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)
        
        # 2. Classical Post-Processing Layers
        self.fc1 = nn.Linear(n_qubits, 16)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.q_layer(x)   # Output from Quantum Circuit
        x = self.fc1(x)       # Classical Dense
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)       # Final prediction
        return self.sigmoid(x)

model = HybridModel().to(device)
criterion = nn.BCELoss() # Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=0.005)

# ==========================================
# 5. TRAINING LOOP
# ==========================================
print("Starting Training...")
epochs = 10 

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Accuracy Calc
        predicted = (outputs > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss/len(train_loader):.4f} | Accuracy: {epoch_acc:.2f}%")

# ==========================================
# 6. EVALUATION & SAVING
# ==========================================
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        predicted = (outputs > 0.5).float()
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

print(f"\nFinal Test Accuracy: {100 * test_correct / test_total:.2f}%")

# SAVE MODEL WEIGHTS
torch.save(model.state_dict(), 'qml_model_weights.pth')
print("\n✅ Model weights saved as 'qml_model_weights.pth'")
print("✅ DOWNLOAD ALL .pkl AND .pth FILES NOW.")

Preprocessing: Scaling and PCA...
✅ Preprocessors saved.
Building Hybrid Quantum Model...
Starting Training...
Epoch [1/10] Loss: 0.1439 | Accuracy: 97.20%
Epoch [2/10] Loss: 0.0857 | Accuracy: 97.43%
Epoch [3/10] Loss: 0.0793 | Accuracy: 97.43%
Epoch [4/10] Loss: 0.0748 | Accuracy: 97.43%
Epoch [5/10] Loss: 0.0724 | Accuracy: 97.43%
Epoch [6/10] Loss: 0.0675 | Accuracy: 97.43%
Epoch [7/10] Loss: 0.0613 | Accuracy: 97.43%
Epoch [8/10] Loss: 0.0559 | Accuracy: 97.43%
Epoch [9/10] Loss: 0.0529 | Accuracy: 97.43%
Epoch [10/10] Loss: 0.0492 | Accuracy: 97.43%

Final Test Accuracy: 97.53%

✅ Model weights saved as 'qml_model_weights.pth'
✅ DOWNLOAD ALL .pkl AND .pth FILES NOW.
